# 99 — End-to-End MLE Demo

Demostración integral del proyecto de Machine Learning Engineering.

## Objetivos

- Ejecutar el flujo completo desde datos crudos hasta predicciones.
- Utilizar exclusivamente los módulos de `src/`.
- Generar artefactos reproducibles.
- Verificar que el modelo guardado produce los mismos resultados.
- Crear un resumen técnico y de negocio.
- Servir como demostración final del primer curso.


## 1. Flujo general

```text
CSV
 ↓
Carga y validación
 ↓
Separación X / y
 ↓
Preprocesamiento
 ↓
Entrenamiento
 ↓
Evaluación
 ↓
Persistencia
 ↓
Recarga
 ↓
Predicción batch
 ↓
Reportes
```


## 2. Configuración del entorno

En Google Colab:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


## 3. Importaciones


In [ ]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from src.data import load_customer_data
from src.features import split_features_target
from src.models import (
    load_model,
    predict_customers,
    save_training_artifacts,
    train_model,
)
from src.utils import get_project_path, get_logger

logger = get_logger("notebook.end_to_end_demo")


## 4. Definición de rutas


In [ ]:
DATA_PATH = get_project_path(
    "data",
    "raw",
    "customer_churn.csv",
)

MODEL_PATH = get_project_path(
    "artifacts",
    "models",
    "end_to_end_churn_pipeline.joblib",
    create_parent=True,
)

METRICS_PATH = get_project_path(
    "reports",
    "metrics",
    "end_to_end_metrics.json",
    create_parent=True,
)

PREDICTIONS_PATH = get_project_path(
    "reports",
    "predictions",
    "end_to_end_predictions.csv",
    create_parent=True,
)

SUMMARY_PATH = get_project_path(
    "reports",
    "metrics",
    "end_to_end_summary.json",
    create_parent=True,
)

for path in [
    DATA_PATH,
    MODEL_PATH,
    METRICS_PATH,
    PREDICTIONS_PATH,
    SUMMARY_PATH,
]:
    print(path)


## 5. Carga y validación de datos


In [ ]:
start_total = time.perf_counter()

df = load_customer_data(
    DATA_PATH,
    validate=True,
)

logger.info(
    "Dataset loaded with shape %s",
    df.shape,
)

df.head()


In [ ]:
data_summary = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_ids": int(
        df["customer_id"].duplicated().sum()
    ),
    "churn_rate": float(
        df["churn"].mean()
    ),
}

data_summary


## 6. Separación de variables


In [ ]:
X, y = split_features_target(df)

print(f"Feature matrix: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target mean: {y.mean():.2%}")


## 7. Entrenamiento


In [ ]:
training_start = time.perf_counter()

training_result = train_model(
    df,
    test_size=0.20,
    random_state=42,
)

training_seconds = (
    time.perf_counter()
    - training_start
)

training_result.metrics


In [ ]:
print(
    f"Train rows: {training_result.train_rows:,}"
)
print(
    f"Test rows: {training_result.test_rows:,}"
)
print(
    f"Training seconds: {training_seconds:.4f}"
)


## 8. Guardado de artefactos


In [ ]:
save_training_artifacts(
    training_result,
    model_path=MODEL_PATH,
    metrics_path=METRICS_PATH,
)

print(f"Model saved: {MODEL_PATH.exists()}")
print(f"Metrics saved: {METRICS_PATH.exists()}")


## 9. Recarga del modelo


In [ ]:
loaded_pipeline = load_model(
    MODEL_PATH
)

print(type(loaded_pipeline))
print(
    loaded_pipeline.named_steps.keys()
)


## 10. Comparación antes y después de serializar


In [ ]:
sample_features = X.head(25)

original_probabilities = (
    training_result.pipeline.predict_proba(
        sample_features
    )[:, 1]
)

loaded_probabilities = (
    loaded_pipeline.predict_proba(
        sample_features
    )[:, 1]
)

serialization_difference = np.abs(
    original_probabilities
    - loaded_probabilities
)

pd.Series(
    serialization_difference
).describe()


In [ ]:
serialization_consistent = np.allclose(
    original_probabilities,
    loaded_probabilities,
)

serialization_consistent


## 11. Preparación del dataset de inferencia


In [ ]:
inference_df = df.drop(
    columns=["churn"]
).copy()

print(inference_df.shape)
inference_df.head()


## 12. Predicción batch


In [ ]:
prediction_start = time.perf_counter()

predictions = predict_customers(
    loaded_pipeline,
    inference_df,
    threshold=0.50,
)

prediction_seconds = (
    time.perf_counter()
    - prediction_start
)

predictions.head()


In [ ]:
print(
    f"Predicted rows: {len(predictions):,}"
)
print(
    f"Prediction seconds: {prediction_seconds:.4f}"
)
print(
    "Rows per second: "
    f"{len(predictions) / prediction_seconds:,.2f}"
)


## 13. Enriquecimiento de resultados


In [ ]:
business_context = inference_df[
    [
        "customer_id",
        "region",
        "customer_segment",
        "contract_type",
        "monthly_fee",
        "tenure_months",
        "support_calls",
        "complaints",
        "last_payment_delay",
    ]
].copy()

final_predictions = (
    business_context
    .merge(
        predictions,
        on="customer_id",
        how="left",
        validate="one_to_one",
    )
)

final_predictions.head()


## 14. Bandas de riesgo


In [ ]:
final_predictions["risk_band"] = pd.cut(
    final_predictions[
        "churn_probability"
    ],
    bins=[
        -np.inf,
        0.35,
        0.55,
        0.75,
        np.inf,
    ],
    labels=[
        "Bajo",
        "Medio",
        "Alto",
        "Muy alto",
    ],
)

final_predictions[
    "risk_band"
].value_counts()


## 15. Priorización para retención


In [ ]:
final_predictions[
    "priority_score"
] = (
    final_predictions[
        "churn_probability"
    ]
    * final_predictions[
        "monthly_fee"
    ]
)

priority_list = (
    final_predictions
    .sort_values(
        "priority_score",
        ascending=False,
    )
)

priority_list.head(20)


## 16. Resumen de negocio


In [ ]:
business_summary = {
    "total_customers": int(
        len(final_predictions)
    ),
    "predicted_churn_customers": int(
        final_predictions[
            "churn_prediction"
        ].sum()
    ),
    "predicted_churn_rate": float(
        final_predictions[
            "churn_prediction"
        ].mean()
    ),
    "average_probability": float(
        final_predictions[
            "churn_probability"
        ].mean()
    ),
    "monthly_fee_at_risk": float(
        final_predictions.loc[
            final_predictions[
                "churn_prediction"
            ] == 1,
            "monthly_fee",
        ].sum()
    ),
    "very_high_risk_customers": int(
        (
            final_predictions[
                "risk_band"
            ] == "Muy alto"
        ).sum()
    ),
}

business_summary


## 17. Validaciones end-to-end


In [ ]:
validation_checks = {
    "model_file_exists": MODEL_PATH.exists(),
    "metrics_file_exists": METRICS_PATH.exists(),
    "serialization_consistent": bool(
        serialization_consistent
    ),
    "prediction_row_count_matches": (
        len(predictions)
        == len(inference_df)
    ),
    "probabilities_in_range": bool(
        predictions[
            "churn_probability"
        ].between(0, 1).all()
    ),
    "predictions_are_binary": bool(
        set(
            predictions[
                "churn_prediction"
            ].unique()
        ).issubset({0, 1})
    ),
    "customer_ids_unique": bool(
        predictions[
            "customer_id"
        ].is_unique
    ),
}

validation_checks


In [ ]:
if not all(
    validation_checks.values()
):
    raise RuntimeError(
        "End-to-end validation failed."
    )

print("All end-to-end checks passed.")


## 18. Exportación de predicciones


In [ ]:
priority_list.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(PREDICTIONS_PATH)


## 19. Resumen técnico


In [ ]:
total_seconds = (
    time.perf_counter()
    - start_total
)

technical_summary = {
    "training_seconds": float(
        training_seconds
    ),
    "prediction_seconds": float(
        prediction_seconds
    ),
    "total_seconds": float(
        total_seconds
    ),
    "prediction_rows_per_second": float(
        len(predictions)
        / prediction_seconds
    ),
    "model_class": type(
        loaded_pipeline.named_steps[
            "classifier"
        ]
    ).__name__,
    "pipeline_steps": list(
        loaded_pipeline.named_steps.keys()
    ),
    "metrics": training_result.metrics,
}

technical_summary


## 20. Exportación del resumen integral


In [ ]:
end_to_end_summary = {
    "data": data_summary,
    "business": business_summary,
    "technical": technical_summary,
    "validation": validation_checks,
    "artifacts": {
        "model": str(MODEL_PATH),
        "metrics": str(METRICS_PATH),
        "predictions": str(
            PREDICTIONS_PATH
        ),
    },
}

SUMMARY_PATH.write_text(
    json.dumps(
        end_to_end_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(SUMMARY_PATH)


## 21. Verificación de archivos


In [ ]:
generated_files = pd.DataFrame({
    "artifact": [
        "model",
        "metrics",
        "predictions",
        "summary",
    ],
    "path": [
        MODEL_PATH,
        METRICS_PATH,
        PREDICTIONS_PATH,
        SUMMARY_PATH,
    ],
})

generated_files["exists"] = (
    generated_files[
        "path"
    ].apply(
        lambda path: Path(path).exists()
    )
)

generated_files["size_bytes"] = (
    generated_files[
        "path"
    ].apply(
        lambda path: (
            Path(path).stat().st_size
            if Path(path).exists()
            else 0
        )
    )
)

generated_files


## 22. Lectura de artefactos


In [ ]:
metrics_payload = json.loads(
    METRICS_PATH.read_text(
        encoding="utf-8"
    )
)

summary_payload = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

print(metrics_payload.keys())
print(summary_payload.keys())


## 23. Ejecución mediante CLI


In [ ]:
import subprocess

cli_process = subprocess.run(
    [
        sys.executable,
        "main.py",
        "train",
        "--data",
        str(DATA_PATH),
        "--model-output",
        str(
            get_project_path(
                "artifacts",
                "models",
                "cli_churn_pipeline.joblib",
                create_parent=True,
            )
        ),
        "--metrics-output",
        str(
            get_project_path(
                "reports",
                "metrics",
                "cli_training_metrics.json",
                create_parent=True,
            )
        ),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(cli_process.stdout)

if cli_process.returncode != 0:
    print(cli_process.stderr)


## 24. Ejecución de pruebas


In [ ]:
test_process = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(test_process.stdout)

if test_process.returncode != 0:
    print(test_process.stderr)


## 25. Resultado final


In [ ]:
project_ready = (
    all(
        validation_checks.values()
    )
    and cli_process.returncode == 0
    and test_process.returncode == 0
)

print(
    "PROJECT READY FOR NEXT COURSE"
    if project_ready
    else "PROJECT REQUIRES REVIEW"
)


## 26. Preguntas para estudiantes

1. ¿Qué componentes se guardan dentro del pipeline?
2. ¿Por qué se compara el modelo antes y después de serializar?
3. ¿Qué diferencia existe entre una métrica técnica y una métrica de negocio?
4. ¿Qué artefactos deberían versionarse con Git y cuáles con DVC?
5. ¿Qué parte del flujo será utilizada por FastAPI?
6. ¿Qué validaciones deberían ejecutarse automáticamente en CI/CD?
7. ¿Qué cambiaría para ejecutar este proceso diariamente?


## 27. Checklist de cierre

- [ ] Se cargaron y validaron los datos.
- [ ] Se entrenó el pipeline.
- [ ] Se calcularon métricas.
- [ ] Se guardó el modelo.
- [ ] Se recargó el artefacto.
- [ ] Se verificó consistencia.
- [ ] Se generaron predicciones batch.
- [ ] Se crearon bandas de riesgo.
- [ ] Se priorizaron clientes.
- [ ] Se exportaron resultados.
- [ ] Se generó el resumen integral.
- [ ] Se ejecutó el CLI.
- [ ] Se ejecutaron pruebas.
- [ ] El proyecto fue aprobado.


## Resultado esperado

Este notebook demuestra que el proyecto puede ejecutarse de extremo a extremo
sin depender de lógica manual dispersa en celdas.

El mismo pipeline será reutilizado posteriormente por:

- FastAPI;
- Docker;
- AWS;
- Prometheus;
- procesos de reentrenamiento.
